# E3 — Calibration Under Population Shift

This notebook evaluates **uncalibrated → sigmoid → isotonic** probability estimates under:

1. **E3-IID:** repeated stratified outer CV;
2. **E3-LOSO:** Leave-One-Student-Type-Out population shift.

The primary model is the **group-blind Logistic Regression** carried forward from E1B/E2.

## Core design

For every outer training set, a 3-fold inner CV generates **out-of-fold base probabilities**.  
Sigmoid and isotonic calibrators are fitted only on those inner OOF predictions. The base model is then refitted on the complete outer-training data and applied to the untouched outer test set. For LOSO, the held-out `Student_Type` is never used to fit either the classifier or the calibrator.

## Actual E3-IID global results from this run

| method       |   roc_auc |   pr_auc |   brier |   brier_skill_score |     nll |   ece_uniform |   ece_quantile |   mce_uniform |   mce_quantile |   calibration_intercept |   calibration_slope |
|:-------------|----------:|---------:|--------:|--------------------:|--------:|--------------:|---------------:|--------------:|---------------:|------------------------:|--------------------:|
| isotonic     |   0.86494 |  0.74942 | 0.12983 |             0.38164 | 0.40921 |       0.00211 |        0.00191 |       0.00591 |        0.00375 |                -0.00766 |             0.9878  |
| sigmoid      |   0.86575 |  0.75192 | 0.12975 |             0.38202 | 0.40866 |       0.01014 |        0.0096  |       0.02332 |        0.01981 |                 0.00184 |             1.00322 |
| uncalibrated |   0.86575 |  0.75193 | 0.12974 |             0.38206 | 0.40865 |       0.00966 |        0.00941 |       0.02182 |        0.01958 |                -0.00121 |             0.99856 |

## Actual E3-LOSO results from this run

| heldout_group   | method       |   brier |   brier_skill_score |     nll |   ece_uniform |   ece_quantile |   mce_uniform |   mce_quantile |   calibration_intercept |   calibration_slope |
|:----------------|:-------------|--------:|--------------------:|--------:|--------------:|---------------:|--------------:|---------------:|------------------------:|--------------------:|
| college         | uncalibrated | 0.13428 |             0.37743 | 0.42112 |       0.00886 |        0.00972 |       0.02867 |        0.01999 |                 0.02288 |             0.99758 |
| college         | sigmoid      | 0.13429 |             0.37738 | 0.42112 |       0.00912 |        0.00985 |       0.02527 |        0.0211  |                 0.02471 |             1.00213 |
| college         | isotonic     | 0.13453 |             0.37628 | 0.42605 |       0.01029 |        0.00521 |       0.02599 |        0.01925 |                 0.00041 |             0.95539 |
| school          | uncalibrated | 0.10989 |             0.38529 | 0.35641 |       0.02034 |        0.01811 |       0.06292 |        0.04813 |                -0.06718 |             0.98611 |
| school          | sigmoid      | 0.10991 |             0.38519 | 0.35641 |       0.01886 |        0.01843 |       0.05401 |        0.04906 |                -0.06257 |             0.99387 |
| school          | isotonic     | 0.10993 |             0.38505 | 0.35581 |       0.01248 |        0.01139 |       0.04828 |        0.04181 |                -0.04087 |             1.01648 |
| working_student | uncalibrated | 0.1451  |             0.41167 | 0.44781 |       0.0134  |        0.01168 |       0.04441 |        0.02209 |                 0.01326 |             0.97626 |
| working_student | sigmoid      | 0.14509 |             0.4117  | 0.4478  |       0.01354 |        0.01151 |       0.0455  |        0.02105 |                 0.01739 |             0.98023 |
| working_student | isotonic     | 0.14549 |             0.41011 | 0.44912 |       0.01883 |        0.0161  |       0.03744 |        0.03461 |                 0.00088 |             0.94513 |

## Weighted / worst-group LOSO summary

| method       |   roc_auc_weighted |   pr_auc_weighted |   brier_weighted |   bss_weighted |   nll_weighted |   ece_uniform_weighted |   ece_quantile_weighted |   mce_uniform_worst |   mce_quantile_worst |   worst_group_abs_intercept |   worst_group_abs_slope_error |
|:-------------|-------------------:|------------------:|-----------------:|---------------:|---------------:|-----------------------:|------------------------:|--------------------:|---------------------:|----------------------------:|------------------------------:|
| isotonic     |            0.85906 |           0.72473 |          0.1306  |        0.3852  |        0.41321 |                0.01253 |                 0.00892 |             0.04828 |              0.04181 |                     0.04087 |                       0.05487 |
| sigmoid      |            0.85965 |           0.73566 |          0.13039 |        0.38616 |        0.41037 |                0.01242 |                 0.01231 |             0.05401 |              0.04906 |                     0.06257 |                       0.01977 |
| uncalibrated |            0.85965 |           0.73566 |          0.13038 |        0.38621 |        0.41037 |                0.01262 |                 0.01219 |             0.06292 |              0.04813 |                     0.06718 |                       0.02374 |

> Associations and probability estimates are benchmark-model results, not causal or clinical claims.


In [ ]:
from pathlib import Path
import platform, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss
from scipy.special import expit, logit
from scipy.optimize import minimize

SEED=42
TARGET="Stress_Level"
GROUP="Student_Type"
N_BINS=10
EPS=1e-6
DATA_PATH=next((p for p in [
    Path("/mnt/data/student-lifestyle-and-stress-dataset.csv"),
    Path("../data/student-lifestyle-and-stress-dataset.csv"),
    Path("student-lifestyle-and-stress-dataset.csv")
] if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Edit DATA_PATH.")
RESULT_DIR=Path("../outputs/E3")
RESULT_DIR.mkdir(exist_ok=True)
df=pd.read_csv(DATA_PATH)
print(df.shape)


In [ ]:
NUMERIC_FEATURES=[
    "Sleep_Hours","Study_Hours","Social_Media_Hours",
    "Attendance","Exam_Pressure","Family_Support"
]
CATEGORICAL_FEATURES=["Month"]
FEATURES=NUMERIC_FEATURES+CATEGORICAL_FEATURES

def make_base_lr():
    prep=ColumnTransformer([
        ("num",Pipeline([
            ("imputer",SimpleImputer(strategy="median")),
            ("scaler",StandardScaler()),
        ]),NUMERIC_FEATURES),
        ("cat",Pipeline([
            ("imputer",SimpleImputer(strategy="most_frequent")),
            ("ohe",OneHotEncoder(handle_unknown="ignore",sparse_output=False)),
        ]),CATEGORICAL_FEATURES),
    ],sparse_threshold=0)
    return Pipeline([
        ("prep",prep),
        ("model",LogisticRegression(max_iter=2000,solver="lbfgs",random_state=SEED)),
    ])

def inner_oof_base_probs(X,y,n_splits=3):
    cv=StratifiedKFold(n_splits=n_splits,shuffle=True,random_state=SEED)
    out=np.full(len(y),np.nan)
    for tr,va in cv.split(X,y):
        m=make_base_lr()
        m.fit(X.iloc[tr],y.iloc[tr])
        out[va]=m.predict_proba(X.iloc[va])[:,1]
    return out

def fit_three(X_train,y_train,X_test):
    Xr=X_train.reset_index(drop=True)
    yr=y_train.reset_index(drop=True)
    inner_p=inner_oof_base_probs(Xr,yr,3)

    xlog=logit(np.clip(inner_p,EPS,1-EPS)).reshape(-1,1)
    sig=LogisticRegression(C=1e6,solver="lbfgs",max_iter=2000)
    sig.fit(xlog,yr)

    iso=IsotonicRegression(out_of_bounds="clip",y_min=0,y_max=1)
    iso.fit(inner_p,yr)

    base=make_base_lr()
    base.fit(X_train,y_train)
    raw=base.predict_proba(X_test)[:,1]

    sigmoid=sig.predict_proba(logit(np.clip(raw,EPS,1-EPS)).reshape(-1,1))[:,1]
    isotonic=np.clip(iso.predict(raw),EPS,1-EPS)

    return {"uncalibrated":raw,"sigmoid":sigmoid,"isotonic":isotonic}


In [ ]:
def calibration_bins(y,p,n_bins=10,strategy="uniform"):
    y=np.asarray(y).astype(int)
    p=np.clip(np.asarray(p,float),0,1)
    edges=(np.linspace(0,1,n_bins+1) if strategy=="uniform"
           else np.unique(np.quantile(p,np.linspace(0,1,n_bins+1))))
    if len(edges)<2:
        edges=np.array([0.,1.])
    else:
        edges[0]=min(edges[0],0); edges[-1]=max(edges[-1],1)
    ids=np.digitize(p,edges[1:-1],right=True)
    rows=[]
    for b in range(len(edges)-1):
        mask=ids==b
        if mask.sum():
            mp=float(p[mask].mean()); er=float(y[mask].mean())
            rows.append({"bin":b+1,"n":int(mask.sum()),"mean_pred":mp,
                         "event_rate":er,"abs_gap":abs(er-mp)})
    return pd.DataFrame(rows)

def ece_mce(y,p,n_bins=10,strategy="uniform"):
    bins=calibration_bins(y,p,n_bins,strategy)
    ece=float(((bins["n"]/bins["n"].sum())*bins["abs_gap"]).sum())
    return ece,float(bins["abs_gap"].max())

def calib_intercept_slope(y,p):
    y=np.asarray(y).astype(float)
    p=np.clip(np.asarray(p,float),EPS,1-EPS)
    x=logit(p)
    def obj(beta):
        q=np.clip(expit(beta[0]+beta[1]*x),EPS,1-EPS)
        return -np.mean(y*np.log(q)+(1-y)*np.log(1-q))
    res=minimize(obj,[0.,1.],method="BFGS")
    return float(res.x[0]),float(res.x[1])

def metrics(y,p,reference_prob):
    y=np.asarray(y).astype(int)
    p=np.clip(np.asarray(p,float),EPS,1-EPS)
    ref=np.asarray(reference_prob,float)
    if ref.ndim==0:
        ref=np.full(len(y),float(ref))
    bs=brier_score_loss(y,p)
    bs_ref=float(np.mean((y-ref)**2))
    eu,mu=ece_mce(y,p,N_BINS,"uniform")
    eq,mq=ece_mce(y,p,N_BINS,"quantile")
    ci,cs=calib_intercept_slope(y,p)
    return {
        "roc_auc":roc_auc_score(y,p),
        "pr_auc":average_precision_score(y,p),
        "brier":bs,
        "brier_reference":bs_ref,
        "brier_skill_score":1-bs/bs_ref,
        "nll":log_loss(y,p,labels=[0,1]),
        "ece_uniform":eu,"ece_quantile":eq,
        "mce_uniform":mu,"mce_quantile":mq,
        "calibration_intercept":ci,"calibration_slope":cs,
    }


## E3-IID — repeated outer CV with inner OOF calibration

In [ ]:
X=df[FEATURES].copy()
y=df[TARGET].astype(int).copy()
outer=RepeatedStratifiedKFold(n_splits=5,n_repeats=2,random_state=SEED)

fold_rows=[]
pred_rows=[]
for sid,(tr,te) in enumerate(outer.split(X,y),1):
    rep=(sid-1)//5+1
    fold=(sid-1)%5+1
    Xtr,Xte=X.iloc[tr],X.iloc[te]
    ytr,yte=y.iloc[tr],y.iloc[te]
    prev=float(ytr.mean())

    for method,p in fit_three(Xtr,ytr,Xte).items():
        fold_rows.append({
            "method":method,"repeat":rep,"fold":fold,
            "n_train":len(tr),"n_test":len(te),"train_prevalence":prev,
            **metrics(yte,p,prev)
        })
        for idx,yt,pr in zip(te,yte.to_numpy(),p):
            pred_rows.append({
                "row_id":int(idx),"repeat":rep,"fold":fold,
                "method":method,"y_true":int(yt),"prob":float(pr),
                "reference_prob":prev,
                "Student_Type":df.loc[idx,GROUP] if pd.notna(df.loc[idx,GROUP]) else "MISSING"
            })

iid_fold=pd.DataFrame(fold_rows)
iid_predictions=pd.DataFrame(pred_rows)
display(iid_fold.groupby("method").mean(numeric_only=True))


In [ ]:
iid_oof=(iid_predictions
    .groupby(["row_id","method","y_true","Student_Type"],as_index=False)
    .agg(prob=("prob","mean"),reference_prob=("reference_prob","mean")))

global_rows=[]
subgroup_rows=[]
for method,part in iid_oof.groupby("method"):
    global_rows.append({"method":method,"n":len(part),
                        "prevalence":part.y_true.mean(),
                        **metrics(part.y_true,part.prob,part.reference_prob)})

for (method,grp),part in iid_oof.groupby(["method","Student_Type"]):
    subgroup_rows.append({"method":method,"Student_Type":grp,"n":len(part),
                          "prevalence":part.y_true.mean(),
                          **metrics(part.y_true,part.prob,part.reference_prob)})

iid_global=pd.DataFrame(global_rows)
iid_subgroup=pd.DataFrame(subgroup_rows)
display(iid_global)
display(iid_subgroup)


## E3-LOSO — calibration on training populations only

In [ ]:
e2=df[df[GROUP].notna()].copy()
rows=[]
preds=[]
for heldout in sorted(e2[GROUP].unique()):
    train=e2[e2[GROUP]!=heldout]
    test=e2[e2[GROUP]==heldout]
    Xtr=train[FEATURES]; ytr=train[TARGET].astype(int)
    Xte=test[FEATURES]; yte=test[TARGET].astype(int)
    prev=float(ytr.mean())

    for method,p in fit_three(Xtr,ytr,Xte).items():
        rows.append({
            "heldout_group":heldout,"method":method,
            "n_train":len(train),"n_test":len(test),
            "train_prevalence":prev,"test_prevalence":float(yte.mean()),
            **metrics(yte,p,prev)
        })
        for idx,yt,pr in zip(test.index,yte.to_numpy(),p):
            preds.append({"row_id":int(idx),"heldout_group":heldout,
                          "method":method,"y_true":int(yt),"prob":float(pr),
                          "reference_prob":prev})

loso=pd.DataFrame(rows)
loso_predictions=pd.DataFrame(preds)
display(loso)


In [ ]:
def plot_reliability(data,title,strategy="quantile"):
    fig,ax=plt.subplots(figsize=(6,5))
    ax.plot([0,1],[0,1],linestyle="--",linewidth=1,label="Perfect calibration")
    for method in ["uncalibrated","sigmoid","isotonic"]:
        part=data[data["method"]==method]
        bins=calibration_bins(part.y_true,part.prob,N_BINS,strategy)
        ax.plot(bins.mean_pred,bins.event_rate,marker="o",label=method)
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed high-stress rate")
    ax.set_title(title)
    ax.legend()
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    plt.tight_layout()
    plt.show()

plot_reliability(iid_oof,"E3-IID — Global reliability")

for grp in sorted(loso_predictions.heldout_group.unique()):
    plot_reliability(
        loso_predictions[loso_predictions.heldout_group==grp],
        f"E3-LOSO — Unseen {grp}"
    )


## Interpretation rule

- Lower is better: **Brier, NLL, ECE, MCE**.
- Higher is better: **Brier Skill Score**.
- Calibration intercept should approach **0**.
- Calibration slope should approach **1**.
- Calibration must be judged jointly across metrics and held-out groups.
- Isotonic can improve ECE while worsening NLL/Brier or discrimination because it is more flexible and can create ties.
